# Detecção de Páginas Faltantes

Verifica quais datas estão com imagens de páginas faltando em uma coleção de jornais.

**Formato esperado dos arquivos:** `YYYY-MM-DD_PAGINA.jpg`  
Exemplo: `2012-06-04_002.jpg`

O resultado é salvo em um arquivo `.txt` com o relatório completo.

## Configuração

- **`INPUT_FOLDER`**: pasta com as imagens (pode ter subpastas — o script varre tudo)
- **`OUTPUT_TXT`**: caminho do relatório de saída
- **`REQUIRED_PAGES`**: páginas esperadas por data. Se `None`, detecta lacunas automaticamente na sequência encontrada.
- **`START_DATE` / `END_DATE`**: intervalo de datas esperado *(opcional)*. Se informado, datas completamente ausentes (sem nenhum arquivo) também aparecem no relatório.

In [ ]:
INPUT_FOLDER = "caminho/para/as/imagens"  # pasta com os JPGs
OUTPUT_TXT = "missing_pages_report.txt" # arquivo de saída
REQUIRED_PAGES = [2, 3] # páginas esperadas por data (ou None para detecção automática)

START_DATE = None  # ex: "2003-01-01" (deixe None para ignorar)
END_DATE = None  # ex: "2013-12-31" (deixe None para ignorar)

## Varredura das imagens

In [ ]:
import os
import re
from collections import defaultdict
from datetime import datetime, timedelta

# Varre a pasta (e subpastas) coletando data e página de cada imagem
pages_by_date = defaultdict(set)
unreadable = []

for root, _, files in os.walk(INPUT_FOLDER):
    for filename in files:
        if not filename.lower().endswith((".jpg", ".jpeg")):
            continue
        match = re.match(r"(\d{4}-\d{2}-\d{2})_(\d+)", filename)
        if match:
            date_str = match.group(1)
            page = int(match.group(2))
            pages_by_date[date_str].add(page)
        else:
            unreadable.append(filename)

all_dates = sorted(pages_by_date.keys())
print(f"Datas encontradas: {len(all_dates)}")
if all_dates:
    print(f"Intervalo: {all_dates[0]} a {all_dates[-1]}")
print(f"Arquivos sem data/página legível no nome: {len(unreadable)}")
if unreadable:
    print("  Exemplos:", unreadable[:5])

## Detecção de páginas faltantes e datas ausentes

In [ ]:
# Páginas faltantes em datas que existem (mas incompletas)
missing_pages = {}

for date_str in all_dates:
    present = pages_by_date[date_str]

    if REQUIRED_PAGES is not None:
        absent = sorted(p for p in REQUIRED_PAGES if p not in present)
    else:
        min_p, max_p = min(present), max(present)
        absent = sorted(p for p in range(min_p, max_p + 1) if p not in present)

    if absent:
        missing_pages[date_str] = absent

# Datas completamente ausentes (nenhum arquivo) dentro do intervalo definido
absent_dates = []

if START_DATE and END_DATE:
    start = datetime.strptime(START_DATE, "%Y-%m-%d")
    end = datetime.strptime(END_DATE,   "%Y-%m-%d")
    current = start
    while current <= end:
        date_str = current.strftime("%Y-%m-%d")
        if date_str not in pages_by_date:
            absent_dates.append(date_str)
        current += timedelta(days=1)

total_missing_pages = sum(len(v) for v in missing_pages.values())

print(f"Datas com páginas faltantes (parcialmente): {len(missing_pages)}")
print(f"Total de páginas faltantes: {total_missing_pages}")
if START_DATE and END_DATE:
    print(f"Datas completamente ausentes no intervalo: {len(absent_dates)}")

## Salvar relatório

In [ ]:
lines = []
lines.append("RELATÓRIO DE PÁGINAS FALTANTES")
lines.append("=" * 50)
lines.append(f"Pasta: {os.path.abspath(INPUT_FOLDER)}")
lines.append(f"Gerado em: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
if REQUIRED_PAGES:
    lines.append(f"Páginas esperadas por data: {REQUIRED_PAGES}")
else:
    lines.append("Páginas esperadas: detecção automática de lacunas")
if START_DATE and END_DATE:
    lines.append(f"Intervalo definido: {START_DATE} a {END_DATE}")

lines.append("")
lines.append("RESUMO")
lines.append("-" * 30)
lines.append(f"Total de datas encontradas: {len(all_dates)}")
lines.append(f"Datas com páginas faltantes (parcial): {len(missing_pages)}")
lines.append(f"Total de páginas faltantes: {total_missing_pages}")
if START_DATE and END_DATE:
    lines.append(f"Datas completamente ausentes: {len(absent_dates)}")

# Seção: datas parcialmente incompletas
if missing_pages:
    lines.append("")
    lines.append("DATAS COM PÁGINAS FALTANTES (parcialmente)")
    lines.append("-" * 30)
    for date_str, absent in sorted(missing_pages.items()):
        pages_str = ", ".join(str(p) for p in absent)
        lines.append(f"[{date_str}] Páginas faltantes: {pages_str}")
else:
    lines.append("")
    lines.append("Nenhuma data com páginas faltantes.")

# Seção: datas completamente ausentes
if START_DATE and END_DATE:
    lines.append("")
    if absent_dates:
        lines.append("DATAS COMPLETAMENTE AUSENTES")
        lines.append("-" * 30)
        for date_str in absent_dates:
            lines.append(date_str)
    else:
        lines.append("Nenhuma data completamente ausente no intervalo.")

report = "\n".join(lines)

with open(OUTPUT_TXT, "w", encoding="utf-8") as f:
    f.write(report)

print(report)